# TranscriptFormer cell embeddings — Open Problems batch correction

This notebook is the TranscriptFormer counterpart of `batch_corr_op.ipynb`.
It embeds the same four datasets with the human `tf-sapiens` checkpoint and
evaluates those embeddings with the same scIB batch-correction metrics.

Datasets included: `dkd`, `gtex_v9`, `hypomap`, and `mouse_pancreas_atlas`.
`immune_cell_atlas` and `tabula_sapiens` are intentionally excluded.


## Recreate the dedicated H100 environment on Jean Zay

Run these commands once from the repository root on a login node. The Python
runtime and environment are portable because H100 nodes do not expose the
`/gpfslocalsup` Python used by V100 nodes. Package downloads occur only during
this explicit setup step; notebook execution itself is fully offline.

```bash
source /etc/profile.d/proxy.sh
export WORK="${WORK:-/lustre/fswork/projects/rech/xeg/$USER}"
export SCRATCH="${SCRATCH:-/lustre/fsn1/projects/rech/xeg/$USER}"
export UV="$HOME/.local/bin/uv"
export UV_CACHE_DIR="$SCRATCH/uv-cache"
export UV_PYTHON_INSTALL_DIR="$WORK/uv-python"
export TF_ENV="$SCRATCH/venvs/transcriptformer-h100-0.6.1"

"$UV" python install 3.11
export PYTHON_311="$WORK/uv-python/cpython-3.11.13-linux-x86_64-gnu/bin/python3.11"
"$UV" venv --python "$PYTHON_311" --relocatable "$TF_ENV"
"$UV" pip install --python "$TF_ENV/bin/python"   transcriptformer==0.6.1 torch==2.5.1 anndata==0.11.4 scanpy==1.11.2   pandas==2.2.2 numpy==2.2.6 scipy==1.15.3 scib-metrics==0.5.10   leidenalg==0.12.0 papermill==2.7.0 ipykernel==6.31.0 triton==3.1.0

mkdir -p "$SCRATCH/scprint_data/setuptools-overlay"
"$UV" pip install --target "$SCRATCH/scprint_data/setuptools-overlay"   setuptools==69.1.1

test -d "$WORK/models/transcriptformer/tf_sapiens"
"$TF_ENV/bin/python" --version
```

TranscriptFormer 0.6.1 pins PyTorch 2.5.1. For these human datasets use
`tf-sapiens`. The model receives raw, unnormalised counts and Ensembl IDs; the
preparation cell validates both conditions before inference.


## One-H100 execution on Jean Zay

The measured configuration uses exactly one H100, mixed precision, batch size
32, and Triton block-mask compilation. Detailed inference progress is written
to one log per dataset under `$SCRATCH`, keeping every notebook result visible.

```bash
mkdir -p "$SCRATCH/scprint_data/slurm_logs"
sbatch   --job-name=tf-op-batch   --output="$SCRATCH/scprint_data/slurm_logs/tf-op-batch-%j.out"   --ntasks-per-node=1   --gres=gpu:1   --constraint=h100   --time=08:00:00   --account=wbg@h100   --nodes=1   --partition=gpu_p6   --hint=nomultithread   --qos=qos_gpu_h100-t3   --cpus-per-task=24   slurm/any_sub.sh   "bash -lc 'export PYTHONDONTWRITEBYTECODE=1; export PYTHONPATH=$SCRATCH/scprint_data/setuptools-overlay; export IPYTHONDIR=$SCRATCH/scprint_data/ipython; export JAX_PLATFORMS=cpu; export XLA_PYTHON_CLIENT_PREALLOCATE=false; export OMP_NUM_THREADS=24; export MKL_NUM_THREADS=24; export OPENBLAS_NUM_THREADS=24; source $SCRATCH/venvs/transcriptformer-h100-0.6.1/bin/activate; python -m papermill --progress-bar --autosave-cell-every 120 -k python3 --log-output notebooks/scPRINT-2-repro-notebooks/batch_corr_op_transcriptformer.ipynb notebooks/scPRINT-2-repro-notebooks/batch_corr_op_transcriptformer.ipynb'"
```

On a 1,024-cell benchmark, this setup completed in 37.3 seconds versus 240.9
seconds for V100 batch size 1. All data and checkpoint paths remain local.


In [ ]:
from pathlib import Path
import gc
import json
import os
import shutil
import subprocess

os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "WANDB_MODE": "offline",
    "WANDB_DISABLED": "true",
    "AWS_EC2_METADATA_DISABLED": "true",
})

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from IPython.display import display
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
from scib_metrics.utils import silhouette_samples
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


In [ ]:
# Fail early if Papermill is using the wrong environment or no GPU was allocated.
subprocess.run(["transcriptformer", "--help"], check=True, stdout=subprocess.DEVNULL)
subprocess.run(["nvidia-smi"], check=True)

import importlib.metadata as metadata

print("TranscriptFormer:", metadata.version("transcriptformer"))
print("Python executable:", shutil.which("python"))


In [ ]:
WORK = Path(os.environ.get("WORK", Path.cwd()))
SCRATCH = Path(os.environ.get("SCRATCH", WORK))

DATA_ROOT = Path("data/temp")
TF_CACHE_ROOT = SCRATCH / "scprint_data"
PREPARED_ROOT = TF_CACHE_ROOT / "transcriptformer_inputs"
OUTPUT_ROOT = TF_CACHE_ROOT / "transcriptformer_outputs"
LOG_ROOT = TF_CACHE_ROOT / "transcriptformer_logs"
PCA_ROOT = TF_CACHE_ROOT / "transcriptformer_pca"
RESULT_ROOT = Path("data/results/transcriptformer")
CHECKPOINT_ROOT = WORK / "models/transcriptformer"
CHECKPOINT_PATH = CHECKPOINT_ROOT / "tf_sapiens"

for directory in (PREPARED_ROOT, OUTPUT_ROOT, LOG_ROOT, PCA_ROOT, RESULT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

TF_BATCH_SIZE = 32
TF_INFERENCE_EXTRA_ARGS = []

datasets = (
    "cellxgene_census/dkd",
    "cellxgene_census/gtex_v9",
    "cellxgene_census/hypomap",
    "cellxgene_census/mouse_pancreas_atlas",
)

assert "cellxgene_census/immune_cell_atlas" not in datasets
assert "cellxgene_census/tabula_sapiens" not in datasets


## Prepare raw-count AnnData inputs

The official TranscriptFormer loader reads `.raw.X` first when it exists. To
make the input unambiguous, this notebook writes a temporary AnnData whose `X`
is the selected raw-count matrix and whose `var["ensembl_id"]` contains Ensembl
IDs. It also samples the matrix to reject negative or non-integer-like values.

Prepared inputs and model outputs are cached, so a restarted Slurm job resumes
at the first unfinished dataset instead of recomputing completed inference.


In [ ]:
def _sample_values(matrix, n_rows=128):
    sample = matrix[: min(n_rows, matrix.shape[0])]
    values = sample.data if sp.issparse(sample) else np.asarray(sample).ravel()
    return np.asarray(values)


def _add_ensembl_ids(adata):
    if "ensembl_id" in adata.var:
        ids = adata.var["ensembl_id"].astype(str)
    elif "feature_id" in adata.var:
        ids = adata.var["feature_id"].astype(str)
    elif pd.Index(adata.var_names.astype(str)).str.startswith("ENS").all():
        ids = pd.Series(adata.var_names.astype(str), index=adata.var_names)
    else:
        raise ValueError(
            "No Ensembl IDs found: expected var['ensembl_id'], "
            "var['feature_id'], or Ensembl-formatted var_names."
        )
    adata.var["ensembl_id"] = ids.to_numpy()


def prepare_transcriptformer_input(name):
    slug = name.rsplit("/", 1)[-1]
    source_path = DATA_ROOT / f"{name}.h5ad"
    processed_path = DATA_ROOT / f"{name}_proc.h5ad"
    prepared_path = PREPARED_ROOT / f"{slug}_raw_counts.h5ad"
    if prepared_path.exists():
        return prepared_path

    if source_path.exists():
        source = sc.read_h5ad(source_path)
        prepared = source.raw.to_adata() if source.raw is not None else source
    elif processed_path.exists():
        print(f"Using cached processed counts from {processed_path}", flush=True)
        source = sc.read_h5ad(processed_path)
        prepared = source
    else:
        raise FileNotFoundError(
            f"{name}: expected {source_path} or {processed_path}; "
            "network downloads are disabled in this notebook."
        )
    prepared.obs = source.obs.copy()
    obs_name_key = "_transcriptformer_input_obs_name"
    if obs_name_key in prepared.obs:
        raise ValueError(f"{name}: reserved obs column already exists: {obs_name_key}")
    prepared.obs[obs_name_key] = prepared.obs_names.astype(str)
    _add_ensembl_ids(prepared)

    values = _sample_values(prepared.X)
    if values.size and (
        np.nanmin(values) < 0
        or not np.allclose(values, np.rint(values), rtol=0, atol=1e-6)
    ):
        raise ValueError(
            f"{name}: TranscriptFormer requires raw, non-negative integer counts."
        )

    # Avoid a second, potentially normalized matrix taking precedence at inference.
    prepared.raw = None
    prepared.write_h5ad(prepared_path, compression="lzf")
    del source, prepared
    gc.collect()
    return prepared_path


def load_pre_integrated_pca(prepared_path, pca_path, expected_obs_names, n_components=50):
    """Load or compute the unintegrated PCA required by scIB's PCR metric."""
    if pca_path.exists():
        pca = np.load(pca_path)
        if pca.shape != (len(expected_obs_names), n_components):
            raise RuntimeError(f"Unexpected cached PCA shape: {pca.shape}")
        print(f"Reusing cached PCA: {pca_path}", flush=True)
        return pca

    baseline = ad.read_h5ad(prepared_path)
    if not np.array_equal(baseline.obs_names.astype(str), expected_obs_names):
        raise RuntimeError("PCA source cell names/order differ from model output")
    sc.pp.normalize_total(baseline, target_sum=1e4)
    sc.pp.log1p(baseline)
    sc.pp.pca(baseline, n_comps=n_components)
    pca = np.asarray(baseline.obsm["X_pca"], dtype=np.float32)
    np.save(pca_path, pca)
    print(f"PCA cached: {pca_path}", flush=True)
    del baseline
    gc.collect()
    return pca


def isolated_labels_low_memory(X, labels, batch, chunk_size=32):
    """Compute scIB isolated-label ASW with smaller, equivalent chunks."""
    pairs = pd.DataFrame({"label": labels, "batch": batch}).drop_duplicates()
    batches_per_label = pairs.groupby("label")["batch"].count()
    isolated = batches_per_label[batches_per_label <= batches_per_label.min()].index
    scores = (silhouette_samples(X, labels, chunk_size=chunk_size) + 1) / 2
    return float(np.mean([scores[np.asarray(labels) == label].mean() for label in isolated]))


def kmeans_nmi_ari_low_memory(X, labels):
    """Compute KMeans NMI/ARI without scib-metrics' N×K×D JAX tensor."""
    model = SklearnKMeans(
        n_clusters=len(np.unique(labels)),
        init="k-means++",
        n_init=1,
        max_iter=300,
        tol=1e-4,
        random_state=0,
        copy_x=False,
        algorithm="lloyd",
    )
    predicted = model.fit_predict(X)
    return {
        "nmi": normalized_mutual_info_score(labels, predicted, average_method="arithmetic"),
        "ari": adjusted_rand_score(labels, predicted),
    }


In [ ]:
if not CHECKPOINT_PATH.is_dir():
    raise FileNotFoundError(
        f"Missing checkpoint directory: {CHECKPOINT_PATH}. "
        "Network downloads are disabled in this notebook."
    )

print("Checkpoint:", CHECKPOINT_PATH)


## Run TranscriptFormer and the scIB benchmark

Each dataset runs in its own code cell and immediately displays its score table.
Verbose TranscriptFormer progress goes to `LOG_ROOT/<dataset>_inference.log`
instead of consuming the notebook output limit. Cell embeddings are read from
`obsm["embeddings"]`; scIB uses `donor_id` as batch and `cell_type` as label.


In [ ]:
def run_transcriptformer_inference(prepared_path, output_path, log_path):
    """Run one-GPU inference while keeping verbose progress out of the notebook."""
    command = [
        "transcriptformer",
        "inference",
        "--checkpoint-path",
        str(CHECKPOINT_PATH),
        "--data-file",
        str(prepared_path),
        "--gene-col-name",
        "ensembl_id",
        "--use-raw",
        "False",
        "--output-path",
        str(OUTPUT_ROOT),
        "--output-filename",
        output_path.name,
        "--emb-type",
        "cell",
        "--device",
        "cuda",
        "--num-gpus",
        "1",
        "--precision",
        "16-mixed",
        "--batch-size",
        str(TF_BATCH_SIZE),
        "--oom-dataloader",
        "--n-data-workers",
        "2",
        *TF_INFERENCE_EXTRA_ARGS,
    ]
    print(f"Inference log: {log_path}", flush=True)
    try:
        with log_path.open("w") as log_file:
            subprocess.run(
                command,
                check=True,
                cwd=LOG_ROOT,
                stdout=log_file,
                stderr=subprocess.STDOUT,
            )
    except subprocess.CalledProcessError:
        tail = log_path.read_text(errors="replace").splitlines()[-40:]
        print("\n".join(tail), flush=True)
        raise


def benchmark_dataset(name):
    """Infer and score one cached OpenProblems dataset, returning its scIB table."""
    slug = name.rsplit("/", 1)[-1]
    output_path = OUTPUT_ROOT / f"{slug}_tf_sapiens_embeddings.h5ad"
    log_path = LOG_ROOT / f"{slug}_inference.log"
    score_path = RESULT_ROOT / f"{slug}_scib.csv"
    print(f"Dataset: {name}", flush=True)

    if score_path.exists():
        print(f"Reusing completed scores: {score_path}", flush=True)
        return pd.read_csv(score_path, index_col=0)

    prepared_path = prepare_transcriptformer_input(name)
    if not output_path.exists():
        run_transcriptformer_inference(prepared_path, output_path, log_path)
    else:
        print(f"Reusing completed embeddings: {output_path}", flush=True)

    embedded = ad.read_h5ad(output_path)
    prepared_backed = ad.read_h5ad(prepared_path, backed="r")
    prepared_obs_names = prepared_backed.obs_names.astype(str).to_numpy(copy=True)
    prepared_backed.file.close()
    obs_name_key = "_transcriptformer_input_obs_name"
    if obs_name_key not in embedded.obs:
        raise KeyError(f"{name}: output obs is missing {obs_name_key!r}")
    embedded_obs_names = embedded.obs.pop(obs_name_key).astype(str).to_numpy()
    if not np.array_equal(embedded_obs_names, prepared_obs_names):
        raise RuntimeError(
            f"{name}: TranscriptFormer output cell names/order differ from input; "
            "refusing to align embeddings positionally."
        )
    embedded.obs_names = embedded_obs_names
    if "embeddings" not in embedded.obsm:
        raise KeyError(f"{name}: output has no obsm['embeddings'] field")
    for required_obs in ("donor_id", "cell_type"):
        if required_obs not in embedded.obs:
            raise KeyError(f"{name}: output obs is missing {required_obs!r}")

    embedding = np.asarray(embedded.obsm.pop("embeddings"), dtype=np.float32)
    embedded.obsm["transcriptformer_emb"] = embedding
    pca_path = PCA_ROOT / f"{slug}_X_pca.npy"
    embedded.obsm["X_pca"] = load_pre_integrated_pca(
        prepared_path, pca_path, embedded.obs_names.astype(str).to_numpy()
    )
    benchmark = Benchmarker(
        embedded,
        batch_key="donor_id",
        label_key="cell_type",
        embedding_obsm_keys=["transcriptformer_emb"],
        pre_integrated_embedding_obsm_key="X_pca",
        bio_conservation_metrics=BioConservation(
            isolated_labels=False,
            nmi_ari_cluster_labels_kmeans=False,
            silhouette_label={"chunk_size": 32},
        ),
        batch_correction_metrics=BatchCorrection(
            bras={"chunk_size": 32}
        ),
        n_jobs=2,
    )
    kmeans_scores = kmeans_nmi_ari_low_memory(
        embedding, embedded.obs["cell_type"].to_numpy()
    )
    benchmark._results.loc["nmi_ari_cluster_labels_kmeans_nmi", "transcriptformer_emb"] = kmeans_scores["nmi"]
    benchmark._results.loc["nmi_ari_cluster_labels_kmeans_nmi", "Metric Type"] = "Bio conservation"
    benchmark._results.loc["nmi_ari_cluster_labels_kmeans_ari", "transcriptformer_emb"] = kmeans_scores["ari"]
    benchmark._results.loc["nmi_ari_cluster_labels_kmeans_ari", "Metric Type"] = "Bio conservation"
    benchmark.benchmark()
    isolated_score = isolated_labels_low_memory(
        embedding, embedded.obs["cell_type"].to_numpy(), embedded.obs["donor_id"].to_numpy()
    )
    benchmark._results.loc["isolated_labels", "transcriptformer_emb"] = isolated_score
    benchmark._results.loc["isolated_labels", "Metric Type"] = "Bio conservation"
    result = benchmark.get_results(min_max_scale=False)
    result.to_csv(score_path)
    print(f"Scores written: {score_path}", flush=True)

    del embedded, benchmark
    gc.collect()
    return result


### DKD


In [ ]:
dkd_result = benchmark_dataset("cellxgene_census/dkd")
display(dkd_result)


### GTEx v9


In [ ]:
gtex_v9_result = benchmark_dataset("cellxgene_census/gtex_v9")
display(gtex_v9_result)


### HypoMap


In [ ]:
hypomap_result = benchmark_dataset("cellxgene_census/hypomap")
display(hypomap_result)


### Mouse pancreas atlas


In [ ]:
mouse_pancreas_atlas_result = benchmark_dataset("cellxgene_census/mouse_pancreas_atlas")
display(mouse_pancreas_atlas_result)


In [ ]:
metrics = {
    "cellxgene_census/dkd": dkd_result,
    "cellxgene_census/gtex_v9": gtex_v9_result,
    "cellxgene_census/hypomap": hypomap_result,
    "cellxgene_census/mouse_pancreas_atlas": mouse_pancreas_atlas_result,
}
combined = pd.concat(metrics, names=["dataset", "method"])
combined_path = RESULT_ROOT / "transcriptformer_scib_all_datasets.csv"
combined.to_csv(combined_path)
print(f"Combined scores written: {combined_path}")
display(combined)
